In [22]:
# Célula 1 - Instalar
!pip install streamlit anthropic -q
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

cloudflared: Text file busy


In [23]:
!pip install streamlit groq pandas chromadb sentence-transformers -q

In [24]:
%%writefile app.py
import streamlit as st
import pandas as pd
from groq import Groq
import chromadb
from sentence_transformers import SentenceTransformer
import hashlib

st.title("🤖 IA com RAG - Gandalf")
st.caption("Busca vetorial — aguenta qualquer tamanho")

api_key = st.text_input("API Key do Groq:", type="password")

# Modelos (carrega uma vez)
@st.cache_resource
def carregar_modelos():
    embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
    chroma_client = chromadb.Client()
    return embedding_model, chroma_client

embedding_model, chroma_client = carregar_modelos()

uploaded_files = st.file_uploader(
    "📂 Envie seus CSVs ou Excel",
    type=["csv", "xlsx", "xls"],
    accept_multiple_files=True
)

def indexar_dataframe(df, nome_arquivo, collection):
    """Divide o CSV em chunks e indexa no ChromaDB"""
    chunk_size = 50  # linhas por chunk
    chunks = []
    ids = []

    for i in range(0, len(df), chunk_size):
        chunk = df.iloc[i:i+chunk_size]
        texto = f"Arquivo: {nome_arquivo}\nLinhas {i} a {i+len(chunk)}:\n{chunk.to_string()}"
        chunk_id = hashlib.md5(texto.encode()).hexdigest()
        chunks.append(texto)
        ids.append(chunk_id)

    # Gera embeddings e salva
    embeddings = embedding_model.encode(chunks).tolist()
    collection.add(documents=chunks, embeddings=embeddings, ids=ids)
    return len(chunks)

if uploaded_files:
    # Cria ou recupera collection
    collection_name = "datasets"
    try:
        collection = chroma_client.get_collection(collection_name)
    except:
        collection = chroma_client.create_collection(collection_name)

    for file in uploaded_files:
        file_hash = hashlib.md5(file.name.encode()).hexdigest()

        if f"indexed_{file_hash}" not in st.session_state:
            with st.spinner(f"Indexando {file.name}..."):
                if file.name.endswith(".csv"):
                    df = pd.read_csv(file)
                else:
                    df = pd.read_excel(file)

                n_chunks = indexar_dataframe(df, file.name, collection)
                st.session_state[f"indexed_{file_hash}"] = True
                st.success(f"✅ {file.name} indexado em {n_chunks} blocos ({df.shape[0]} linhas)")
        else:
            st.info(f"✅ {file.name} já está indexado")

    st.divider()

    if "messages" not in st.session_state:
        st.session_state.messages = []

    for msg in st.session_state.messages:
        with st.chat_message(msg["role"]):
            st.write(msg["content"])

    if prompt := st.chat_input("olá sou Gandalf, seu Mentor do Dinheiro, como posso te ajudar hoje?"):
        if not api_key:
            st.error("Insira sua API key primeiro!")
        else:
            st.session_state.messages.append({"role": "user", "content": prompt})
            with st.chat_message("user"):
                st.write(prompt)

            # Busca os chunks mais relevantes
            query_embedding = embedding_model.encode([prompt]).tolist()
            resultados = collection.query(
                query_embeddings=query_embedding,
                n_results=5  # pega os 5 pedaços mais relevantes
            )
            contexto = "\n\n".join(resultados["documents"][0])

            system_prompt = f"""Você é um analista de dados experiente.
Responda a pergunta do usuário com base nos trechos do dataset abaixo.
Se não encontrar a informação, diga que não está nos dados disponíveis.

TRECHOS RELEVANTES:
{contexto}
"""
            client = Groq(api_key=api_key)
            mensagens = [
                {"role": "system", "content": system_prompt},
                *st.session_state.messages
            ]

            with st.chat_message("assistant"):
                with st.spinner("Entendi pequeno mestre,vou verificar para você"):
                    response = client.chat.completions.create(
                        model="llama-3.3-70b-versatile",
                        messages=mensagens,
                        max_tokens=2048
                    )
                    reply = response.choices[0].message.content
                    st.write(reply)

            st.session_state.messages.append({"role": "assistant", "content": reply})

else:
    st.info("⬆️ Envie seus arquivos para começar")

Overwriting app.py


In [25]:
# Célula 3 - Rodar tudo
import subprocess, threading, time, re

def run_streamlit():
    subprocess.run(["streamlit", "run", "app.py",
                    "--server.port=8501",
                    "--server.headless=true"])

threading.Thread(target=run_streamlit, daemon=True).start()
time.sleep(4)

# Inicia o túnel Cloudflare
tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stderr=subprocess.PIPE, stdout=subprocess.PIPE
)

# Captura a URL gerada
for line in tunnel.stderr:
    line = line.decode()
    if "trycloudflare.com" in line:
        url = re.search(r'https://\S+\.trycloudflare\.com', line)
        if url:
            print(f"✅ Acesse aqui: {url.group()}")
            break

✅ Acesse aqui: https://doll-jackson-ship-kijiji.trycloudflare.com


In [26]:
"""
import pandas as pd

# Dataset 1 - Orçamento
orcamento = pd.DataFrame([
    {"categoria": "Moradia", "tipo": "necessidade", "percentual_recomendado": 30, "exemplo": "Aluguel/financiamento", "dica": "Não ultrapasse 30% da renda"},
    {"categoria": "Alimentação", "tipo": "necessidade", "percentual_recomendado": 15, "exemplo": "Supermercado/restaurante", "dica": "Cozinhar em casa economiza 40%"},
    {"categoria": "Transporte", "tipo": "necessidade", "percentual_recomendado": 10, "exemplo": "Combustível/transporte público", "dica": "Considere transporte público"},
    {"categoria": "Lazer", "tipo": "desejo", "percentual_recomendado": 10, "exemplo": "Streaming/passeios", "dica": "Lazer saudável é necessário"},
    {"categoria": "Investimentos", "tipo": "prioridade", "percentual_recomendado": 20, "exemplo": "Poupança/Tesouro Direto", "dica": "Pague-se primeiro"},
    {"categoria": "Dívidas", "tipo": "atenção", "percentual_recomendado": 15, "exemplo": "Cartão/empréstimo", "dica": "Quite as de maior juros primeiro"},
])
orcamento.to_csv("dataset_orcamento.csv", index=False)

# Dataset 2 - Juros
juros = pd.DataFrame([
    {"tipo_juro": "Juros simples", "definicao": "Calculado só sobre o valor inicial", "formula": "J = P x i x t", "exemplo_pratico": "Carnê de loja", "armadilha": "Parece barato mas não é"},
    {"tipo_juro": "Juros compostos", "definicao": "Juros sobre juros", "formula": "M = P x (1+i)^t", "exemplo_pratico": "Poupança e cartão de crédito", "armadilha": "No cartão corrói rápido"},
    {"tipo_juro": "CDI", "definicao": "Taxa interbancária de referência", "formula": "Referência para investimentos", "exemplo_pratico": "CDB 100% CDI", "armadilha": "Comparar sempre com inflação"},
    {"tipo_juro": "Selic", "definicao": "Taxa básica da economia", "formula": "Definida pelo Banco Central", "exemplo_pratico": "Tesouro Selic", "armadilha": "Varia com política monetária"},
    {"tipo_juro": "Rotativo cartão", "definicao": "Juros mais altos do Brasil", "formula": "~400% ao ano", "exemplo_pratico": "Pagar mínimo da fatura", "armadilha": "Armadilha mais perigosa"},
])
juros.to_csv("dataset_juros.csv", index=False)

# Dataset 3 - Investimentos
investimentos = pd.DataFrame([
    {"investimento": "Poupança", "tipo": "Renda fixa", "risco": "Baixo", "liquidez": "Diária", "rentabilidade": "~6% ao ano", "valor_minimo": "R$1", "indicado_para": "Iniciantes", "obs": "Pode perder para inflação"},
    {"investimento": "Tesouro Selic", "tipo": "Renda fixa", "risco": "Baixo", "liquidez": "D+1", "rentabilidade": "Selic atual", "valor_minimo": "R$30", "indicado_para": "Reserva de emergência", "obs": "Mais seguro do Brasil"},
    {"investimento": "CDB", "tipo": "Renda fixa", "risco": "Baixo/médio", "liquidez": "Varia", "rentabilidade": "100-120% CDI", "valor_minimo": "R$1", "indicado_para": "Iniciantes", "obs": "Verificar cobertura FGC"},
    {"investimento": "LCI/LCA", "tipo": "Renda fixa", "risco": "Baixo", "liquidez": "Carência", "rentabilidade": "CDI isento IR", "valor_minimo": "R$1000", "indicado_para": "Quem paga IR", "obs": "Isento de imposto de renda"},
    {"investimento": "Ações", "tipo": "Renda variável", "risco": "Alto", "liquidez": "D+3", "rentabilidade": "Variável", "valor_minimo": "R$1", "indicado_para": "Perfil arrojado", "obs": "Estudar antes de investir"},
    {"investimento": "FIIs", "tipo": "Renda variável", "risco": "Médio", "liquidez": "D+3", "rentabilidade": "Dividendos mensais", "valor_minimo": "R$10", "indicado_para": "Renda passiva", "obs": "Diversificar entre fundos"},
])
investimentos.to_csv("dataset_investimentos.csv", index=False)

# Dataset 4 - Conceitos
conceitos = pd.DataFrame([
    {"conceito": "Inflação", "definicao": "Alta geral dos preços", "exemplo": "R$100 comprava mais em 2020", "erro_comum": "Deixar dinheiro parado", "como_evitar": "Investir acima da inflação"},
    {"conceito": "Reserva de emergência", "definicao": "Dinheiro para imprevistos", "exemplo": "Perda de emprego/doença", "erro_comum": "Não ter nenhuma", "como_evitar": "Guardar 6x os gastos mensais"},
    {"conceito": "Juros do rotativo", "definicao": "Maior taxa do Brasil", "exemplo": "Pagar mínimo da fatura", "erro_comum": "Achar que é normal", "como_evitar": "Sempre pagar fatura total"},
    {"conceito": "Financiamento", "definicao": "Compra parcelada com juros", "exemplo": "Carro/imóvel financiado", "erro_comum": "Olhar só a parcela", "como_evitar": "Calcular o total pago"},
    {"conceito": "Previdência privada", "definicao": "Investimento para aposentadoria", "exemplo": "PGBL e VGBL", "erro_comum": "Depender só do INSS", "como_evitar": "Começar cedo"},
    {"conceito": "FGC", "definicao": "Garante depósitos até R$250k", "exemplo": "CDB/poupança se banco quebrar", "erro_comum": "Não saber que existe", "como_evitar": "Diversificar entre bancos"},
])
conceitos.to_csv("dataset_conceitos.csv", index=False)

print("✅ 4 datasets criados com sucesso!")
"""

'\nimport pandas as pd\n\n# Dataset 1 - Orçamento\norcamento = pd.DataFrame([\n    {"categoria": "Moradia", "tipo": "necessidade", "percentual_recomendado": 30, "exemplo": "Aluguel/financiamento", "dica": "Não ultrapasse 30% da renda"},\n    {"categoria": "Alimentação", "tipo": "necessidade", "percentual_recomendado": 15, "exemplo": "Supermercado/restaurante", "dica": "Cozinhar em casa economiza 40%"},\n    {"categoria": "Transporte", "tipo": "necessidade", "percentual_recomendado": 10, "exemplo": "Combustível/transporte público", "dica": "Considere transporte público"},\n    {"categoria": "Lazer", "tipo": "desejo", "percentual_recomendado": 10, "exemplo": "Streaming/passeios", "dica": "Lazer saudável é necessário"},\n    {"categoria": "Investimentos", "tipo": "prioridade", "percentual_recomendado": 20, "exemplo": "Poupança/Tesouro Direto", "dica": "Pague-se primeiro"},\n    {"categoria": "Dívidas", "tipo": "atenção", "percentual_recomendado": 15, "exemplo": "Cartão/empréstimo", "dica"